# Notebook 24 — Continuous Universality Density Fields

This notebook turns the residual universality manifold into continuous density fields.

It is designed to run in either:

1. **Colab from GitHub** — clones `thinkthoughts/residue-manifold-learning` when needed.
2. **Local repo checkout** — uses the current repo root.
3. **Fallback mode** — reconstructs a small deterministic manifold if prior result CSVs are unavailable.

Primary inputs, when available:

- `results/residual_universality_embedding.csv`
- `results/residual_pca_embedding.csv`
- `results/residual_classification_feature_matrix.csv`
- `results/residual_geometry_features.csv`
- `results/ood_transfer_embedding.csv`
- `results/ood_transfer_summary.csv`

Primary outputs:

- `figures/24_density_known_manifold.png`
- `figures/24_density_ood_overlay.png`
- `figures/24_decision_field_nearest_family.png`
- `figures/24_boundary_confidence_field.png`
- `figures/24_universality_potential_field.png`
- `results/continuous_density_grid.csv`
- `results/continuous_density_summary.csv`
- `docs/notebook_24_summary.md`

In [ ]:
# Optional Colab repo setup
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/thinkthoughts/residue-manifold-learning.git"
REPO_NAME = "residue-manifold-learning"

def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / ".git").exists() or (p / "notebooks").exists() or (p / "results").exists():
            return p
    return start

cwd = Path.cwd()

# In Colab, default cwd is usually /content. Clone repo if needed.
if cwd == Path("/content") or str(cwd).startswith("/content"):
    candidate = Path("/content") / REPO_NAME
    if not candidate.exists():
        print(f"Cloning {REPO_URL} -> {candidate}")
        subprocess.run(["git", "clone", REPO_URL, str(candidate)], check=True)
    os.chdir(candidate)
    REPO_ROOT = candidate
else:
    REPO_ROOT = find_repo_root(cwd)

RESULTS_DIR = REPO_ROOT / "results"
FIGURES_DIR = REPO_ROOT / "figures"
DOCS_DIR = REPO_ROOT / "docs"

for d in [RESULTS_DIR, FIGURES_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("cwd:", Path.cwd())
print("repo root:", REPO_ROOT)
print("available result files:", sorted(p.name for p in RESULTS_DIR.glob("*")))

In [ ]:
# Imports
import math
import json
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from scipy.spatial.distance import cdist
from scipy.ndimage import gaussian_filter

## 1. Load or reconstruct residual manifold inputs

The preferred path is to use prior notebook outputs. If the runtime has no prior CSVs, this notebook reconstructs a deterministic scaffold so the full workflow still runs and exports the expected files.

In [ ]:
# Robust input loader

candidate_files = {
    "universality": RESULTS_DIR / "residual_universality_embedding.csv",
    "pca": RESULTS_DIR / "residual_pca_embedding.csv",
    "feature_matrix": RESULTS_DIR / "residual_classification_feature_matrix.csv",
    "geometry": RESULTS_DIR / "residual_geometry_features.csv",
    "ood_embedding": RESULTS_DIR / "ood_transfer_embedding.csv",
    "ood_summary": RESULTS_DIR / "ood_transfer_summary.csv",
}

for key, path in candidate_files.items():
    print(f"{key:>14}:", path.exists(), path)

known_raw = None
source_label = None

if candidate_files["universality"].exists():
    known_raw = pd.read_csv(candidate_files["universality"])
    source_label = "residual_universality_embedding.csv"
elif candidate_files["pca"].exists():
    known_raw = pd.read_csv(candidate_files["pca"])
    source_label = "residual_pca_embedding.csv"
elif candidate_files["feature_matrix"].exists():
    fm = pd.read_csv(candidate_files["feature_matrix"])
    numeric_cols = [c for c in fm.columns if pd.api.types.is_numeric_dtype(fm[c]) and c not in ["N", "n_modules"]]
    label_col = "topology" if "topology" in fm.columns else "family"
    scaler = StandardScaler()
    X = scaler.fit_transform(fm[numeric_cols].fillna(fm[numeric_cols].median()))
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)
    known_raw = pd.DataFrame({
        "topology": fm[label_col].astype(str),
        "N": fm["N"] if "N" in fm.columns else fm.get("n_modules", np.arange(len(fm))),
        "PC1": coords[:, 0],
        "PC2": coords[:, 1],
    })
    source_label = "PCA from residual_classification_feature_matrix.csv"
elif candidate_files["geometry"].exists():
    geom = pd.read_csv(candidate_files["geometry"])
    label_col = "topology" if "topology" in geom.columns else "family"
    numeric_cols = [c for c in geom.columns if pd.api.types.is_numeric_dtype(geom[c]) and c not in ["N", "n_modules"]]
    scaler = StandardScaler()
    X = scaler.fit_transform(geom[numeric_cols].fillna(geom[numeric_cols].median()))
    pca = PCA(n_components=2)
    coords = pca.fit_transform(X)
    known_raw = pd.DataFrame({
        "topology": geom[label_col].astype(str),
        "N": geom["N"] if "N" in geom.columns else geom.get("n_modules", np.arange(len(geom))),
        "PC1": coords[:, 0],
        "PC2": coords[:, 1],
    })
    source_label = "PCA from residual_geometry_features.csv"

if known_raw is None:
    print("No prior residual manifold CSV found. Using deterministic fallback scaffold.")
    topologies = ["ring lattice", "small world", "Erdős–Rényi", "scale free", "modular clustered"]
    Ns = [16, 32, 64, 128]
    base = {
        "ring lattice": (-2.2, 0.4),
        "small world": (-2.6, 0.9),
        "Erdős–Rényi": (-0.9, -0.1),
        "scale free": (1.5, -0.5),
        "modular clustered": (3.5, 0.1),
    }
    drift = {
        "ring lattice": [(0.4, 1.0), (0.1, 0.1), (-0.3, -0.4), (-0.5, -1.2)],
        "small world": [(-0.5, 2.6), (-0.8, 0.8), (-0.7, -1.4), (-0.6, -2.1)],
        "Erdős–Rényi": [(0.2, 1.0), (0.1, 0.3), (-0.3, -0.8), (-0.5, -1.5)],
        "scale free": [(0.6, 1.5), (-0.1, 0.2), (-0.4, -0.6), (-0.7, -0.9)],
        "modular clustered": [(0.5, 1.1), (0.2, -0.1), (-0.2, -0.7), (-0.4, -1.0)],
    }
    rows = []
    for topo in topologies:
        bx, by = base[topo]
        for N, (dx, dy) in zip(Ns, drift[topo]):
            rows.append({"topology": topo, "N": N, "PC1": bx + dx, "PC2": by + dy})
    known_raw = pd.DataFrame(rows)
    source_label = "deterministic fallback scaffold"

print("source:", source_label)
known_raw.head()

In [ ]:
# Normalize column names and clean labels

def clean_topology_name(x):
    x = str(x).replace("_", " ").strip()
    x = x.replace("erdos renyi", "Erdős–Rényi")
    x = x.replace("Erdos Renyi", "Erdős–Rényi")
    x = x.replace("erdos-renyi", "Erdős–Rényi")
    return x

known = known_raw.copy()

# infer coordinate columns
pc1_candidates = ["PC1", "pc1", "x", "coordinate_1", "manifold_coordinate_1", "residual_manifold_coordinate_1"]
pc2_candidates = ["PC2", "pc2", "y", "coordinate_2", "manifold_coordinate_2", "residual_manifold_coordinate_2"]

pc1 = next((c for c in pc1_candidates if c in known.columns), None)
pc2 = next((c for c in pc2_candidates if c in known.columns), None)

if pc1 is None or pc2 is None:
    numeric_cols = [c for c in known.columns if pd.api.types.is_numeric_dtype(known[c])]
    if len(numeric_cols) < 2:
        raise ValueError("Need at least two numeric columns for manifold coordinates.")
    pc1, pc2 = numeric_cols[:2]

label_col = "topology" if "topology" in known.columns else ("family" if "family" in known.columns else None)
if label_col is None:
    known["topology"] = "unknown"
else:
    known["topology"] = known[label_col].map(clean_topology_name)

if "N" not in known.columns:
    if "n_modules" in known.columns:
        known["N"] = known["n_modules"]
    elif "graph_size" in known.columns:
        known["N"] = known["graph_size"]
    else:
        known["N"] = np.arange(len(known))

known = known.rename(columns={pc1: "PC1", pc2: "PC2"})
known = known[["topology", "N", "PC1", "PC2"]].dropna().copy()
known["N"] = pd.to_numeric(known["N"], errors="coerce").fillna(0).astype(int)
known = known.sort_values(["topology", "N"]).reset_index(drop=True)

known.to_csv(RESULTS_DIR / "residual_density_known_points.csv", index=False)
known

## 2. Build continuous density and nearest-family fields

We estimate a smooth density over the residual manifold using Gaussian radial kernels centered at known points. We also compute a nearest-family field based on topology centroids.

In [ ]:
# Grid construction

pad = 0.75
x_min, x_max = known["PC1"].min() - pad, known["PC1"].max() + pad
y_min, y_max = known["PC2"].min() - pad, known["PC2"].max() + pad

nx, ny = 220, 220
xs = np.linspace(x_min, x_max, nx)
ys = np.linspace(y_min, y_max, ny)
XX, YY = np.meshgrid(xs, ys)
grid = np.column_stack([XX.ravel(), YY.ravel()])

points = known[["PC1", "PC2"]].to_numpy()
topologies = list(known["topology"].drop_duplicates())
centroids = known.groupby("topology")[["PC1", "PC2"]].mean().loc[topologies]

# Bandwidth: fraction of median interpoint distance, safely bounded.
D = pairwise_distances(points)
nonzero = D[D > 0]
bandwidth = float(np.median(nonzero) * 0.45) if len(nonzero) else 1.0
bandwidth = max(bandwidth, 0.35)
print("kernel bandwidth:", bandwidth)

def gaussian_density(grid_xy, centers, h):
    dist2 = cdist(grid_xy, centers) ** 2
    dens = np.exp(-0.5 * dist2 / (h ** 2)).sum(axis=1)
    return dens

total_density = gaussian_density(grid, points, bandwidth)
total_density = total_density / (total_density.max() + 1e-12)

family_densities = {}
for topo in topologies:
    centers = known.loc[known["topology"] == topo, ["PC1", "PC2"]].to_numpy()
    d = gaussian_density(grid, centers, bandwidth)
    family_densities[topo] = d / (d.max() + 1e-12)

centroid_dist = cdist(grid, centroids[["PC1", "PC2"]].to_numpy())
nearest_idx = np.argmin(centroid_dist, axis=1)
nearest_family = np.array(topologies, dtype=object)[nearest_idx]

# confidence = gap between nearest and second-nearest centroid distances, normalized.
sorted_dist = np.sort(centroid_dist, axis=1)
gap = sorted_dist[:, 1] - sorted_dist[:, 0]
confidence = gap / (sorted_dist[:, 1] + 1e-12)
confidence = np.clip(confidence, 0, 1)

# potential field: high near dense known residual manifold, low near uncertain boundaries
potential = total_density * confidence

grid_df = pd.DataFrame({
    "PC1": grid[:, 0],
    "PC2": grid[:, 1],
    "density": total_density,
    "nearest_family": nearest_family,
    "boundary_confidence": confidence,
    "universality_potential": potential,
})

for topo in topologies:
    grid_df[f"density_{topo.replace(' ', '_').replace('–','_')}"] = family_densities[topo]

grid_df.to_csv(RESULTS_DIR / "continuous_density_grid.csv", index=False)
grid_df.head()

In [ ]:
# Summary table
summary_rows = []
for topo in topologies:
    sub = known[known["topology"] == topo]
    c = centroids.loc[topo]
    family_grid = grid_df[grid_df["nearest_family"] == topo]
    summary_rows.append({
        "topology": topo,
        "n_points": len(sub),
        "centroid_PC1": c["PC1"],
        "centroid_PC2": c["PC2"],
        "mean_N": sub["N"].mean(),
        "assigned_grid_fraction": len(family_grid) / len(grid_df),
        "mean_assigned_density": family_grid["density"].mean() if len(family_grid) else np.nan,
        "mean_assigned_confidence": family_grid["boundary_confidence"].mean() if len(family_grid) else np.nan,
        "mean_assigned_potential": family_grid["universality_potential"].mean() if len(family_grid) else np.nan,
    })

density_summary = pd.DataFrame(summary_rows).sort_values("topology")
density_summary.to_csv(RESULTS_DIR / "continuous_density_summary.csv", index=False)
density_summary

## 3. Known manifold density field

In [ ]:
# Figure 1: known manifold density
fig, ax = plt.subplots(figsize=(10, 7))

Z = grid_df["density"].to_numpy().reshape(ny, nx)
im = ax.contourf(XX, YY, Z, levels=24, alpha=0.75)
plt.colorbar(im, ax=ax, label="kernel density")

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=2, label=topo)
    for _, row in sub.iterrows():
        ax.text(row["PC1"], row["PC2"], f"N={int(row['N'])}", fontsize=8, ha="left", va="bottom")

ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.set_title("Continuous density over known residual universality manifold")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "24_density_known_manifold.png", dpi=180)
plt.show()

## 4. Nearest-family decision field

In [ ]:
# Figure 2: nearest-family decision regions
fig, ax = plt.subplots(figsize=(10, 7))

family_to_int = {t: i for i, t in enumerate(topologies)}
region = np.array([family_to_int[x] for x in nearest_family]).reshape(ny, nx)

im = ax.contourf(XX, YY, region, levels=np.arange(len(topologies)+1)-0.5, alpha=0.25)
for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.scatter(sub["PC1"], sub["PC2"], s=70, label=topo)
    c = centroids.loc[topo]
    ax.scatter(c["PC1"], c["PC2"], marker="*", s=220, color="black")
    ax.text(c["PC1"], c["PC2"], topo, fontsize=10, fontweight="bold", ha="left", va="bottom")

# Boundary confidence contours: low confidence near region boundaries.
C = grid_df["boundary_confidence"].to_numpy().reshape(ny, nx)
ax.contour(XX, YY, C, levels=[0.1, 0.2, 0.35], colors="black", linewidths=1, alpha=0.5)

ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.set_title("Nearest-family residual manifold decision field")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "24_decision_field_nearest_family.png", dpi=180)
plt.show()

## 5. Boundary confidence field

In [ ]:
# Figure 3: boundary confidence
fig, ax = plt.subplots(figsize=(10, 7))

C = grid_df["boundary_confidence"].to_numpy().reshape(ny, nx)
im = ax.contourf(XX, YY, C, levels=24, alpha=0.8)
plt.colorbar(im, ax=ax, label="boundary confidence")

ax.scatter(known["PC1"], known["PC2"], s=45, color="black", alpha=0.7)
for topo in topologies:
    c = centroids.loc[topo]
    ax.scatter(c["PC1"], c["PC2"], marker="*", s=220, color="white", edgecolor="black")
    ax.text(c["PC1"], c["PC2"], topo, fontsize=10, fontweight="bold", ha="left", va="bottom")

ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.set_title("Boundary confidence field")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "24_boundary_confidence_field.png", dpi=180)
plt.show()

## 6. Universality potential field

This field combines density and nearest-family confidence. It highlights regions that are both close to observed residual structures and away from ambiguous topology boundaries.

In [ ]:
# Figure 4: universality potential
fig, ax = plt.subplots(figsize=(10, 7))

P = grid_df["universality_potential"].to_numpy().reshape(ny, nx)
im = ax.contourf(XX, YY, P, levels=24, alpha=0.85)
plt.colorbar(im, ax=ax, label="density × boundary confidence")

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.plot(sub["PC1"], sub["PC2"], marker="o", linewidth=2, label=topo)

ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.set_title("Universality potential field")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "24_universality_potential_field.png", dpi=180)
plt.show()

## 7. Optional OOD overlay

If Notebook 22 or 23 outputs are available, this section overlays out-of-distribution residual trajectories on the same continuous field.

In [ ]:
# Optional OOD overlay
ood = None
ood_source = None

if candidate_files["ood_embedding"].exists():
    ood = pd.read_csv(candidate_files["ood_embedding"])
    ood_source = "ood_transfer_embedding.csv"
elif (RESULTS_DIR / "boundary_sweep_embedding.csv").exists():
    ood = pd.read_csv(RESULTS_DIR / "boundary_sweep_embedding.csv")
    ood_source = "boundary_sweep_embedding.csv"

if ood is not None:
    # normalize possible columns
    ood = ood.copy()
    label_col = "ood_family" if "ood_family" in ood.columns else ("family" if "family" in ood.columns else ("sweep" if "sweep" in ood.columns else None))
    xcol = next((c for c in ["PC1", "pc1", "x"] if c in ood.columns), None)
    ycol = next((c for c in ["PC2", "pc2", "y"] if c in ood.columns), None)
    ncol = next((c for c in ["N", "n_modules", "graph_size"] if c in ood.columns), None)
    if label_col and xcol and ycol:
        ood = ood.rename(columns={xcol: "PC1", ycol: "PC2"})
        if ncol:
            ood = ood.rename(columns={ncol: "N"})
        else:
            ood["N"] = np.arange(len(ood))
        ood["family"] = ood[label_col].astype(str)
        print("Loaded OOD:", ood_source, ood.shape)
    else:
        print("Found OOD file but could not infer columns:", ood.columns.tolist())
        ood = None
else:
    print("No OOD embedding found; skipping OOD overlay.")

In [ ]:
# Figure 5: OOD overlay when available
fig, ax = plt.subplots(figsize=(10, 7))

Z = grid_df["density"].to_numpy().reshape(ny, nx)
im = ax.contourf(XX, YY, Z, levels=24, alpha=0.55)
plt.colorbar(im, ax=ax, label="known manifold density")

for topo in topologies:
    sub = known[known["topology"] == topo]
    ax.scatter(sub["PC1"], sub["PC2"], s=60, alpha=0.5, label=topo)

if ood is not None:
    for fam, sub in ood.groupby("family"):
        sub = sub.sort_values("N")
        ax.plot(sub["PC1"], sub["PC2"], marker="x", linestyle="--", linewidth=2, label=f"OOD: {fam}")
        for _, row in sub.iterrows():
            ax.text(row["PC1"], row["PC2"], f"N={int(row['N'])}", fontsize=8, ha="left", va="bottom")
else:
    ax.text(0.5, 0.5, "OOD embedding unavailable", transform=ax.transAxes, ha="center", va="center", fontsize=14)

ax.axhline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.axvline(0, linestyle="--", linewidth=1, color="black", alpha=0.6)
ax.set_title("OOD overlay on continuous residual density")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "24_density_ood_overlay.png", dpi=180)
plt.show()

## 8. Export notebook summary

In [ ]:
# Markdown summary export
summary_md = f"""# Notebook 24 Summary — Continuous Universality Density Fields

Generated: {datetime.utcnow().isoformat()} UTC

## Source

- Input source: `{source_label}`
- Known manifold points: {len(known)}
- Topology families: {", ".join(topologies)}
- Kernel bandwidth: {bandwidth:.4f}

## Main outputs

- `figures/24_density_known_manifold.png`
- `figures/24_density_ood_overlay.png`
- `figures/24_decision_field_nearest_family.png`
- `figures/24_boundary_confidence_field.png`
- `figures/24_universality_potential_field.png`
- `results/continuous_density_grid.csv`
- `results/continuous_density_summary.csv`

## Interpretation

Notebook 24 converts the discrete residual universality manifold into a continuous diagnostic surface.

- High density identifies regions near observed residual trajectories.
- Nearest-family fields define approximate topology basins.
- Low boundary confidence marks transition zones where topology assignment is unstable.
- Universality potential combines density with boundary confidence, highlighting stable residual regions.

This prepares the repo for paper figures that connect discrete residual trajectories to continuous universality fields.
"""

summary_path = DOCS_DIR / "notebook_24_summary.md"
summary_path.write_text(summary_md)
print(summary_path)
print(summary_md)

In [ ]:
# Optional zip download pack for Colab
MAKE_ZIP = False

if MAKE_ZIP:
    zip_path = REPO_ROOT / "notebook_24_outputs.zip"
    wanted = [
        FIGURES_DIR / "24_density_known_manifold.png",
        FIGURES_DIR / "24_density_ood_overlay.png",
        FIGURES_DIR / "24_decision_field_nearest_family.png",
        FIGURES_DIR / "24_boundary_confidence_field.png",
        FIGURES_DIR / "24_universality_potential_field.png",
        RESULTS_DIR / "continuous_density_grid.csv",
        RESULTS_DIR / "continuous_density_summary.csv",
        RESULTS_DIR / "residual_density_known_points.csv",
        DOCS_DIR / "notebook_24_summary.md",
    ]
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for p in wanted:
            if p.exists():
                zf.write(p, p.relative_to(REPO_ROOT))
    print("wrote:", zip_path)
    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception as e:
        print("Download manually:", zip_path)
        print(e)